# Credit Card Fraud Detection

This notebook follows the CS770 project proposal using two fraud-detection dataset sources:

- `creditcard.csv`
- `fraudTrain.csv` plus `fraudTest.csv`

The goal is binary classification: predict whether a credit card transaction is fraud or not fraud.

## Project Plan

The proposal says to compare Random Forest and Logistic Regression, account for class imbalance, use dimensionality reduction with PCA, and report accuracy, precision, recall, F1-score, confusion matrix, ROC curve, and AUC.

Because the two dataset sources have different schemas, this notebook trains and evaluates separate models for each source.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    PrecisionRecallDisplay,
    recall_score,
    RocCurveDisplay,
    roc_auc_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")

## 1. Load the Selected Datasets

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_CANDIDATES = [PROJECT_ROOT / "datasets", PROJECT_ROOT.parent / "datasets"]
DATA_DIR = next(
    (
        path
        for path in DATA_CANDIDATES
        if (path / "creditcard.csv").exists()
        and (path / "fraudTrain.csv").exists()
        and (path / "fraudTest.csv").exists()
    ),
    DATA_CANDIDATES[0],
)

CREDITCARD_PATH = DATA_DIR / "creditcard.csv"
TRAIN_PATH = DATA_DIR / "fraudTrain.csv"
TEST_PATH = DATA_DIR / "fraudTest.csv"

print("Creditcard data path:", CREDITCARD_PATH)
print("Training data path:", TRAIN_PATH)
print("Testing data path:", TEST_PATH)

creditcard_df = pd.read_csv(CREDITCARD_PATH)
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

print("creditcard shape:", creditcard_df.shape)
print("train shape:", train_df.shape)
print("test shape:", test_df.shape)
train_df.head()

In [ ]:
print("fraudTrain/fraudTest schema")
train_df.info()

print("\ncreditcard.csv schema")
creditcard_df.info()

## 2. Initial Analysis: Class Imbalance

Fraud detection datasets are usually highly imbalanced, meaning most transactions are not fraud. Because of that, accuracy alone can be misleading. Precision, recall, F1-score, PR-AUC, ROC-AUC, and the confusion matrix are more useful.

In [ ]:
def show_class_balance(df, label="dataset", target_column="is_fraud"):
    counts = df[target_column].value_counts().sort_index()
    percents = df[target_column].value_counts(normalize=True).sort_index() * 100
    balance = pd.DataFrame({"count": counts, "percent": percents.round(4)})
    balance.index = ["not fraud", "fraud"]
    print(label)
    display(balance)
    return balance

train_balance = show_class_balance(train_df, "fraudTrain.csv", target_column="is_fraud")
test_balance = show_class_balance(test_df, "fraudTest.csv", target_column="is_fraud")
creditcard_balance = show_class_balance(creditcard_df, "creditcard.csv", target_column="Class")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_specs = [
    (axes[0], train_df, "is_fraud", "fraudTrain.csv"),
    (axes[1], test_df, "is_fraud", "fraudTest.csv"),
    (axes[2], creditcard_df, "Class", "creditcard.csv"),
]

for ax, df, target, title in plot_specs:
    sns.countplot(data=df, x=target, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("fraud label")
    ax.set_ylabel("transactions")
    ax.set_xticklabels(["not fraud", "fraud"])

plt.tight_layout()

## 3. Experiment 1: `fraudTrain.csv` and `fraudTest.csv`

The original transaction dataset contains identifiers and personally identifying text fields that are not good model inputs. This section keeps useful transaction features and creates date-based features like transaction hour, day of week, and cardholder age.

In [ ]:
def add_time_features(df):
    df = df.copy()
    df["trans_date_trans_time"] = pd.to_datetime(df["trans_date_trans_time"], errors="coerce")
    df["dob"] = pd.to_datetime(df["dob"], errors="coerce")

    df["transaction_hour"] = df["trans_date_trans_time"].dt.hour
    df["transaction_dayofweek"] = df["trans_date_trans_time"].dt.dayofweek
    df["transaction_month"] = df["trans_date_trans_time"].dt.month
    df["age"] = (df["trans_date_trans_time"] - df["dob"]).dt.days / 365.25

    return df

train_features_df = add_time_features(train_df)
test_features_df = add_time_features(test_df)

feature_columns = [
    "amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "transaction_hour",
    "transaction_dayofweek",
    "transaction_month",
    "age",
    "category",
    "gender",
]

target_column = "is_fraud"

X_train_full = train_features_df[feature_columns]
y_train_full = train_features_df[target_column]
X_test_full = test_features_df[feature_columns]
y_test_full = test_features_df[target_column]

X_train_full.head()

In [ ]:
USE_SAMPLE = False
TRAIN_SAMPLE_SIZE = 50_000
TEST_SAMPLE_SIZE = 25_000

def stratified_sample(X, y, sample_size, random_state=67):
    if (not USE_SAMPLE) or len(X) <= sample_size:
        return X.copy(), y.copy()

    _, X_sample, _, y_sample = train_test_split(
        X,
        y,
        test_size=sample_size,
        stratify=y,
        random_state=random_state,
    )
    return X_sample.reset_index(drop=True), y_sample.reset_index(drop=True)

X_train, y_train = stratified_sample(X_train_full, y_train_full, TRAIN_SAMPLE_SIZE)
X_test, y_test = stratified_sample(X_test_full, y_test_full, TEST_SAMPLE_SIZE)

print("training shape:", X_train.shape)
print("testing shape:", X_test.shape)
show_class_balance(pd.DataFrame({target_column: y_train}), "Model training set")
show_class_balance(pd.DataFrame({target_column: y_test}), "Model testing set")

In [ ]:
def build_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", drop="first", sparse=False)

numeric_features = [
    "amt",
    "lat",
    "long",
    "city_pop",
    "merch_lat",
    "merch_long",
    "transaction_hour",
    "transaction_dayofweek",
    "transaction_month",
    "age",
]

categorical_features = ["category", "gender"]


def build_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]),
                numeric_features,
            ),
            (
                "categorical",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", build_one_hot_encoder()),
                ]),
                categorical_features,
            ),
        ],
        remainder="drop",
        sparse_threshold=0,
    )


def build_pca():
    return PCA(n_components=0.95, random_state=67)

In [ ]:
logistic_regression = Pipeline(steps=[
    ("preprocess", build_preprocessor()),
    ("pca", build_pca()),
    ("model", LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=67,
        solver="lbfgs",
    )),
])

random_forest = Pipeline(steps=[
    ("preprocess", build_preprocessor()),
    ("pca", build_pca()),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=67,
    )),
])

In [ ]:
models = {
    "Logistic Regression": logistic_regression,
    "Random Forest": random_forest,
}

fitted_models = {}
for model_name, model in models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    fitted_models[model_name] = model
    print(f"Finished {model_name}")

In [ ]:
for model_name, model in fitted_models.items():
    n_components = model.named_steps["pca"].n_components_
    explained_variance = model.named_steps["pca"].explained_variance_ratio_.sum()
    print(f"{model_name}: PCA kept {n_components} components explaining {explained_variance:.2%} variance")

In [ ]:
def evaluate_model(model_name, model, X_test, y_test, threshold=0.50):
    y_prob = model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "model": model_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_test, y_prob),
        "pr_auc": average_precision_score(y_test, y_prob),
    }

    print(f"\n{model_name}")
    print(classification_report(y_test, y_pred, target_names=["not fraud", "fraud"], zero_division=0))

    return metrics, y_pred, y_prob

results = []
predictions = {}

for model_name, model in fitted_models.items():
    metrics, y_pred, y_prob = evaluate_model(model_name, model, X_test, y_test)
    results.append(metrics)
    predictions[model_name] = {"y_pred": y_pred, "y_prob": y_prob}

results_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False)
results_df

In [ ]:
fig, axes = plt.subplots(1, len(fitted_models), figsize=(6 * len(fitted_models), 5))
if len(fitted_models) == 1:
    axes = [axes]

for ax, (model_name, preds) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test, preds["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["not fraud", "fraud"])
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(model_name)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, model in fitted_models.items():
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax, name=model_name)
ax.set_title("ROC Curve")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, model in fitted_models.items():
    PrecisionRecallDisplay.from_estimator(model, X_test, y_test, ax=ax, name=model_name)
ax.set_title("Precision-Recall Curve")
plt.tight_layout()

## 8. Threshold Tuning

Fraud detection usually cares about catching fraud cases without creating too many false alarms. The default threshold of 0.50 is not always best, so this section compares thresholds using precision, recall, and F1-score.

In [ ]:
threshold_rows = []
thresholds = np.arange(0.05, 0.96, 0.05)

for model_name, preds in predictions.items():
    y_prob = preds["y_prob"]
    for threshold in thresholds:
        y_pred_threshold = (y_prob >= threshold).astype(int)
        threshold_rows.append({
            "model": model_name,
            "threshold": threshold,
            "precision": precision_score(y_test, y_pred_threshold, zero_division=0),
            "recall": recall_score(y_test, y_pred_threshold, zero_division=0),
            "f1": f1_score(y_test, y_pred_threshold, zero_division=0),
        })

threshold_df = pd.DataFrame(threshold_rows)
threshold_df.sort_values(["model", "f1"], ascending=[True, False]).groupby("model").head(5)

In [ ]:
fig, axes = plt.subplots(1, len(fitted_models), figsize=(6 * len(fitted_models), 4), sharey=True)
if len(fitted_models) == 1:
    axes = [axes]

for ax, model_name in zip(axes, fitted_models.keys()):
    subset = threshold_df[threshold_df["model"] == model_name]
    ax.plot(subset["threshold"], subset["precision"], label="precision")
    ax.plot(subset["threshold"], subset["recall"], label="recall")
    ax.plot(subset["threshold"], subset["f1"], label="f1")
    ax.set_title(model_name)
    ax.set_xlabel("threshold")
    ax.set_ylabel("score")
    ax.legend()

plt.tight_layout()

In [ ]:
rf_grid = Pipeline(steps=[
    ("preprocess", build_preprocessor()),
    ("pca", build_pca()),
    ("model", RandomForestClassifier(
        class_weight="balanced",
        n_jobs=-1,
        random_state=67,
    )),
])
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_leaf": [1, 2, 5],
}
grid_search = GridSearchCV(
    estimator=rf_grid,
    param_grid=param_grid,
    scoring="roc_auc",
    cv=3,
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train, y_train)
print("best score:", grid_search.best_score_)
print("best params:", grid_search.best_params_)

## 11. Experiment 2: `creditcard.csv`

`creditcard.csv` has a different structure from the `fraudTrain.csv` and `fraudTest.csv` files. It already contains mostly numeric PCA-style features named `V1` through `V28`, plus `Time`, `Amount`, and the target column `Class`.

Because this file does not come with a separate test CSV, this section creates a stratified train/test split from `creditcard.csv`.

In [ ]:
creditcard_target = "Class"
creditcard_feature_columns = [column for column in creditcard_df.columns if column != creditcard_target]

X_creditcard = creditcard_df[creditcard_feature_columns]
y_creditcard = creditcard_df[creditcard_target]

X_credit_train, X_credit_test, y_credit_train, y_credit_test = train_test_split(
    X_creditcard,
    y_creditcard,
    test_size=0.20,
    stratify=y_creditcard,
    random_state=RANDOM_STATE,
)

print("creditcard training shape:", X_credit_train.shape)
print("creditcard testing shape:", X_credit_test.shape)
show_class_balance(pd.DataFrame({creditcard_target: y_credit_train}), "creditcard training split", target_column=creditcard_target)
show_class_balance(pd.DataFrame({creditcard_target: y_credit_test}), "creditcard testing split", target_column=creditcard_target)

### Creditcard Preprocessing and Models

The dataset is already numeric, but `Amount` and `Time` are on different scales from the `V` features. The pipeline imputes missing values, scales all features, applies PCA, and trains the same two model types.

In [ ]:
def build_creditcard_pipeline(model):
    return Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=0.95, random_state=RANDOM_STATE)),
        ("model", model),
    ])

creditcard_models = {
    "Creditcard Logistic Regression": build_creditcard_pipeline(LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=RANDOM_STATE,
        solver="lbfgs",
    )),
    "Creditcard Random Forest": build_creditcard_pipeline(RandomForestClassifier(
        class_weight="balanced",
        n_estimators=200,
        max_depth=None,
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )),
}

fitted_creditcard_models = {}
for model_name, model in creditcard_models.items():
    print(f"Training {model_name}...")
    model.fit(X_credit_train, y_credit_train)
    fitted_creditcard_models[model_name] = model
    print(f"Finished {model_name}")

In [ ]:
creditcard_results = []
creditcard_predictions = {}

for model_name, model in fitted_creditcard_models.items():
    metrics, y_pred, y_prob = evaluate_model(model_name, model, X_credit_test, y_credit_test)
    creditcard_results.append(metrics)
    creditcard_predictions[model_name] = {"y_pred": y_pred, "y_prob": y_prob}

creditcard_results_df = pd.DataFrame(creditcard_results).sort_values("pr_auc", ascending=False)
creditcard_results_df

In [ ]:
fig, axes = plt.subplots(1, len(fitted_creditcard_models), figsize=(6 * len(fitted_creditcard_models), 5))
if len(fitted_creditcard_models) == 1:
    axes = [axes]

for ax, (model_name, preds) in zip(axes, creditcard_predictions.items()):
    cm = confusion_matrix(y_credit_test, preds["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["not fraud", "fraud"])
    disp.plot(ax=ax, values_format="d", colorbar=False)
    ax.set_title(model_name)

plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, model in fitted_creditcard_models.items():
    RocCurveDisplay.from_estimator(model, X_credit_test, y_credit_test, ax=ax, name=model_name)
ax.set_title("creditcard.csv ROC Curve")
plt.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for model_name, model in fitted_creditcard_models.items():
    PrecisionRecallDisplay.from_estimator(model, X_credit_test, y_credit_test, ax=ax, name=model_name)
ax.set_title("creditcard.csv Precision-Recall Curve")
plt.tight_layout()

## 12. Compare Both Dataset Sources

Use the transaction results table and the `creditcard.csv` results table separately in the report. The schemas are different, so their scores should not be interpreted as one shared train/test experiment. They are two related fraud-detection experiments using the same modeling approach.

In [ ]:
print("fraudTrain/fraudTest results")
display(results_df)

print("creditcard.csv results")
display(creditcard_results_df)